## Учимся получать эмбеддинги из DINOv2*

*Meta признана экстремистской организацией.

1. Рассчитываем cls_token dinov2 для всех ббоксов, полученных с помощью YOLO
2. Складываем их в ChromaDB
3. Папку ChromaDB кладём в s3

In [1]:
import sys
sys.path.append('..')
sys.path.append('../..')
sys.path.append('../../..')

from tqdm import tqdm
from pathlib import Path
import os
from PIL import Image
import chromadb
from dotenv import load_dotenv
load_dotenv()

from sneakers_hse.inference.embedding_model import DINOv2Embedder
from sneakers_hse.data.utils.s3_tools import S3Client

In [2]:
PROJECT_ROOT_PATH = Path('/Users/alexansemyachkin/Desktop/studies/hse/sneakers-hse')
PATH_TO_YOLO_OUTPUT = PROJECT_ROOT_PATH / 'data/03_yolo_preprocessed/search-dataset-images'

# соберём все нужные файлы заранее
all_files = []
for root, _, files in os.walk(PATH_TO_YOLO_OUTPUT):
    for file in files:
        local_path = os.path.join(root, file)
        relative_path = os.path.relpath(local_path, PATH_TO_YOLO_OUTPUT)
        if not relative_path.endswith(".DS_Store"):
            all_files.append((root, file, relative_path))

len(all_files)

11265

In [3]:
# Инициализируем эмбеддер и векторную БД
embedder = DINOv2Embedder(device='mps')

client = chromadb.PersistentClient(
    path=PROJECT_ROOT_PATH / "chroma_db"
)
collection = client.get_or_create_collection(
    "embeddings",
    metadata={"hnsw:space": "cosine"})


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

In [4]:
def embed_bbox(relative_paths: list[str | Path]):
    image_paths = [PATH_TO_YOLO_OUTPUT / relative_path for relative_path in relative_paths]
    imgs = [Image.open(path) for path in image_paths]
    embeddings = embedder.encode_batch(imgs)
    ids = [str(relative_path) for relative_path in relative_paths]
    paths = relative_paths
    classes = [str(relative_path).split("/")[0] for relative_path in relative_paths]
    collection.add(
        ids=ids,
        embeddings=embeddings.tolist(),
        metadatas=[{
            "path": str(relative_path),
            "class": str(relative_path).split("/")[0]
        } for relative_path in relative_paths])
    return [{
                "path": path,
                "class": cls,
                "embedding": emb.tolist()
            } for emb, path, cls in zip(embeddings, paths, classes)]

embed_bbox(['adidas Кеды VL COURT 3/clothing_0_2.jpeg',
            'Nike Кеды Dunk Low Retro/clothing_0_86.jpeg'])

[{'path': 'adidas Кеды VL COURT 3/clothing_0_2.jpeg',
  'class': 'adidas Кеды VL COURT 3',
  'embedding': [-2.0206799507141113,
   -0.4394451081752777,
   -1.3173210620880127,
   -0.1369943916797638,
   1.117270827293396,
   -2.1784398555755615,
   -1.4885834455490112,
   -1.722230315208435,
   2.8261146545410156,
   0.15741080045700073,
   0.39157596230506897,
   0.6806805729866028,
   -1.876137614250183,
   0.12393775582313538,
   -2.0237443447113037,
   0.4221615791320801,
   -1.0774050951004028,
   -0.98524409532547,
   -1.378481388092041,
   0.9072139263153076,
   -2.7611446380615234,
   0.5929002165794373,
   4.455023765563965,
   0.9065343737602234,
   0.8539847135543823,
   -2.1804094314575195,
   -0.5578349232673645,
   -1.8463164567947388,
   -1.582446813583374,
   2.695230484008789,
   -1.4852039813995361,
   -1.8405556678771973,
   1.2539823055267334,
   -1.0481585264205933,
   -5.559535026550293,
   -0.21280843019485474,
   -1.3973643779754639,
   -2.4748780727386475,
   0

In [5]:
def chunked(iterable, batch_size):
    batch = []
    for item in iterable:
        batch.append(item[2])
        if len(batch) == batch_size:
            yield batch
            batch = []
    if batch:
        yield batch

all_rows = []
# прогресс‑бар с полосой
for relative_paths in tqdm(chunked(iter(all_files), 32), desc="Processing images"):
    all_rows += embed_bbox(relative_paths)

Processing images: 353it [04:52,  1.21it/s]


In [6]:
import polars as pl

df = pl.DataFrame(all_rows)
df.write_parquet(PROJECT_ROOT_PATH / 'data/04_dinov2_embeddings.parquet.gzip')

In [7]:
from dotenv import load_dotenv

load_dotenv()
s3 = S3Client(aws_access_key_id=os.getenv("AWS_ACCESS_KEY"),
              aws_secret_access_key=os.getenv("AWS_SECRET_KEY"))
# s3.upload_folder_to_s3_parallel(
#     bucket_name='sneakers-hse-images-test',
#     s3_prefix='dinov2/chroma_db', # Путь пишем с учётом модели, т.е. каждой модели будет соответстовать отдельный инстанс ChromaDB
#     local_folder=str(PROJECT_ROOT_PATH / 'chroma_db'),
#     max_workers=10
# )

In [8]:
# s3._upload_one(PROJECT_ROOT_PATH / 'data/04_dinov2_embeddings.parquet.gzip', 'sneakers-hse-images-test', s3_key='dinov2/embeddings.parquet.gzip')

### Проверяю, что query работает

In [9]:
image_path = PATH_TO_YOLO_OUTPUT / 'adidas Кеды VL COURT 3/clothing_0_2.jpeg'
img = Image.open(image_path)
embedding = embedder.encode_batch([img])
collection.query(embedding)

{'ids': [['adidas Кеды VL COURT 3/clothing_0_2.jpeg',
   'adidas Кеды VL COURT 3/clothing_0_63.jpeg',
   'adidas Кеды VL COURT 3/clothing_0_144.jpeg',
   'adidas Кеды VL COURT 3/clothing_0_56.jpeg',
   'adidas Кеды VL COURT 3/clothing_0_75.jpeg',
   'Tendance Кеды/clothing_0_19.jpeg',
   'adidas Кеды VL COURT 3/orig_81.jpeg',
   'adidas Кеды VL COURT 3/clothing_0_126.jpeg',
   'Vans Кеды Upland/clothing_0_212.jpeg',
   'adidas Кроссовки GALAXY 7 M/clothing_0_163.jpeg']],
 'embeddings': None,
 'documents': [[None, None, None, None, None, None, None, None, None, None]],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'path': 'adidas Кеды VL COURT 3/clothing_0_2.jpeg',
    'class': 'adidas Кеды VL COURT 3'},
   {'path': 'adidas Кеды VL COURT 3/clothing_0_63.jpeg',
    'class': 'adidas Кеды VL COURT 3'},
   {'class': 'adidas Кеды VL COURT 3',
    'path': 'adidas Кеды VL COURT 3/clothing_0_144.jpeg'},
   {'class': 'adidas Кеды VL COURT 3'

In [10]:
# Проверяю, что и без нормализации косинусное расстояние корректно считается
import numpy as np


a = embedding[0]
b = collection.get('Vans Кеды Upland/clothing_0_212.jpeg', include=['embeddings'])['embeddings'][0]
1 - np.dot(a, b) / np.linalg.norm(a) / np.linalg.norm(b) 

np.float64(0.0915073674400505)

In [11]:
b = collection.get('Vans Кеды Upland/orig_108.jpeg', include=['embeddings'])['embeddings'][0]
1 - np.dot(a, b) / np.linalg.norm(a) / np.linalg.norm(b) 

np.float64(0.11362339646907504)

### Генерирую аугментированные эмбеддинги для train-сплита (для обучения triplet-модели)

In [12]:
import albumentations as A
import numpy as np
import polars as pl
from PIL import Image

# Без Normalize и ToTensorV2 — DINOv2Embedder использует AutoImageProcessor,
# который сам делает resize и normalize
aug_tfms = A.Compose([
    A.RandomResizedCrop(size=(224, 224), scale=(0.7, 1.0), ratio=(0.75, 1.33), p=1.0),
    A.HorizontalFlip(p=0.5),
    A.Affine(
        scale=(0.9, 1.1),
        translate_percent=(0.0, 0.1),
        rotate=(-15, 15),
        shear=(-10, 10),
        p=0.7
    ),
    A.Perspective(scale=(0.02, 0.08), p=0.5),
    A.OneOf([
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    ], p=0.7),
    A.OneOf([
        A.MotionBlur(blur_limit=5),
        A.GaussianBlur(blur_limit=5),
    ], p=0.3),
    A.GaussNoise(std_range=(0.03, 0.08), p=0.3),
    A.CoarseDropout(
        num_holes_range=(1, 2),
        hole_height_range=(11, 34),
        hole_width_range=(11, 34),
        fill=0,
        p=0.5
    ),
    A.ImageCompression(quality_range=(40, 100), p=0.3),
    A.RandomShadow(
        shadow_roi=(0, 0.5, 1, 1),
        num_shadows_limit=(1, 2),
        shadow_intensity_range=(0.3, 0.6),
        p=0.2
    ),
    A.RandomFog(fog_coef_range=(0.1, 0.3), p=0.15),
])

In [13]:
import polars as pl

# Берём только train-изображения — аугментации нужны для обучения, не для retrieval-индекса
train_csv = pl.read_csv(PROJECT_ROOT_PATH / 'data/03_yolo_preprocessed/train_images.csv')
train_paths = set(train_csv['path'].to_list())

N_AUGS = 3  # количество аугментированных копий на каждое изображение

aug_rows = []
train_files = [(root, file, rel) for root, file, rel in all_files if rel in train_paths]

for root, file, relative_path in tqdm(train_files, desc="Augmenting train images"):
    img = Image.open(os.path.join(root, file)).convert("RGB")
    img_np = np.array(img)
    cls = str(relative_path).split("/")[0]

    for aug_idx in range(N_AUGS):
        augmented = aug_tfms(image=img_np)["image"]
        aug_pil = Image.fromarray(augmented)
        emb = embedder.encode_batch([aug_pil])[0]
        aug_rows.append({
            "path": f"{relative_path}_aug_{aug_idx}",
            "class": cls,
            "embedding": emb.tolist()
        })

Augmenting train images: 100%|██████████| 10965/10965 [22:52<00:00,  7.99it/s]


In [14]:
aug_df = pl.DataFrame(aug_rows)
aug_df.write_parquet(PROJECT_ROOT_PATH / 'data/04_dinov2_embeddings_augmented_train.parquet.gzip')
print(f"Сохранено {len(aug_df)} аугментированных эмбеддингов")

Сохранено 32895 аугментированных эмбеддингов


In [16]:
s3._upload_one(PROJECT_ROOT_PATH / 'data/04_dinov2_embeddings_augmented_train.parquet.gzip', 'sneakers-hse-images-test', s3_key='dinov2/embeddings_augmented_train.parquet.gzip')

'dinov2/embeddings_augmented_train.parquet.gzip'